In [ ]:
import numpy as np
from numba import njit
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import h5py

L = 32 # (16)
J = 1
kb = 1

T_array = np.linspace(1.0, 3.5, 100)   

# Parametry Monte Carlo 
n_thermal = 4000      
n_between = 150        
n_samples_per_T = 2000 

@njit(nogil=True)
def delta_energy(spins, i, j):
    neighbours_sum = spins[(i+1)%L, j] + spins[(i-1)%L, j] + spins[i, (j+1)%L] + spins[i, (j-1)%L]
    return 2 * J * neighbours_sum * spins[i,j]

@njit(nogil=True)
def metropolis_sweep(spins, temperature):
    N = spins.shape[0]
    for _ in range(N * N):
        i = np.random.randint(0, N)
        j = np.random.randint(0, N)
        dE = delta_energy(spins, i, j)
        if dE <= 0 or np.random.rand() < np.exp(-dE / (kb * temperature)):
            spins[i, j] = -spins[i, j]

# Funkcja kompilowana przez Numbę tylko dla JEDNEJ temperatury
@njit(nogil=True)
def generate_single_T(temp, n_samples_per_T, n_thermal, n_between):
    # Generujemy losowe spiny - int8!
    spins = np.random.randint(0, 2, size=(L, L)).astype(np.int8) * 2 - 1
    
    # Tworzymy lokalną tablicę na próbki dla tej konkretnej temperatury
    states = np.empty((n_samples_per_T, L, L), dtype=np.int8)
    
    # Rozgrzewka (Termalizacja)
    for _ in range(n_thermal):
        metropolis_sweep(spins, temp)
        
    # Zbieranie próbek
    for s_idx in range(n_samples_per_T):
        for _ in range(n_between):
            metropolis_sweep(spins, temp)
        states[s_idx] = spins.copy()
        
    return states

# Funkcja pomocnicza dla wielowątkowości
def worker(idx):
    temp = T_array[idx]
    states = generate_single_T(temp, n_samples_per_T, n_thermal, n_between)
    return idx, states

if __name__ == "__main__":
    num_T = len(T_array)
    # Główna, ogromna macierz trzymana w RAMie (dla L=32 to zaledwie ~100MB)
    dataset_X = np.empty((num_T, n_samples_per_T, L, L), dtype=np.int8)

    print(f"Rozpoczynam symulacje dla {num_T} temperatur na wszystkich rdzeniach...")
    
    # ThreadPoolExecutor inteligentnie rozrzuci zadania na wszystkie dostępne rdzenie CPU
    with ThreadPoolExecutor() as executor:
        # Odpalamy 100 niezależnych procesów
        futures = [executor.submit(worker, i) for i in range(num_T)]
        
        # as_completed w połączeniu z tqdm daje nam przepiękny pasek postępu!
        for future in tqdm(as_completed(futures), total=num_T, desc="Generowanie MC"):
            idx, states = future.result()
            dataset_X[idx] = states

    print(f"\nWygenerowano pełny zbiór! Kształt: {dataset_X.shape}")
    
    # Zapis do pliku
    file_name = "2ising_mc_data_16.h5"
    with h5py.File(file_name, 'w') as f:
        f.create_dataset('spins', data=dataset_X, compression='gzip', dtype='i1', chunks=True)
        f.create_dataset('temperatures', data=T_array)
    print(f"Dane skompresowane i zapisane do pliku: {file_name}")

Rozpoczynam symulacje dla 100 temperatur na wszystkich rdzeniach...


Generowanie MC: 100%|██████████| 100/100 [00:21<00:00,  4.72it/s]



Wygenerowano pełny zbiór! Kształt: (100, 2000, 16, 16)
Dane skompresowane i zapisane do pliku: 2ising_mc_data_16.h5
